# 01 — Data preparation

This notebook provides the reproducible starting point for the study. It constructs the experimental dataset from four locally stored corpora without downloading data or accessing the network.

**Before running.** Install the project environment, launch Jupyter from the repository root, and place the corpora under `ori_data/` using the layout specified in `docs/data-sources.md`.

**Workflow.** The pipeline audits the source files, parses and filters the messages, resolves duplicates and cross-label conflicts, assigns each similarity group entirely to the train, validation or test split, and constructs a balanced dataset of 12,780 messages. It then generates Detector Input v1.0 and v2.0 from the same frozen samples.

**Outputs.** The pipeline saves the audit records, split manifest and detector representations to `artifacts/data/` for use in Notebooks 02 and 03.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

from phishing_detection import StudyConfig, prepare_data

config = StudyConfig(root=Path.cwd())
config

## Rebuild and audit controls

For a standard run, leave `FORCE_REBUILD=False`. The pipeline reuses valid artefacts when they already exist and builds the dataset automatically when they do not. Set this option to `True` only when intentionally rebuilding all derived data.

With `FULL_RAW_AUDIT=False`, the pipeline performs the default two-level checks of paths, file counts and byte counts. Set this option to `True` when a full per-file SHA-256 audit is required; this audit is substantially slower for the Enron corpus.

In [ ]:
FORCE_REBUILD = False
FULL_RAW_AUDIT = False

data_audit = prepare_data(config, force=FORCE_REBUILD, full_raw_audit=FULL_RAW_AUDIT)
data_audit

In [ ]:
manifest = pq.read_table(config.split_path).to_pandas()
counts = manifest.groupby(['source', 'split']).size().unstack(fill_value=0)
assert len(manifest) == 12_780, 'Unexpected corpus total; inspect the audit before continuing.'
assert manifest['sample_id'].is_unique
assert manifest.groupby('similarity_group_id')['split'].nunique().max() == 1
assert pq.read_metadata(config.detector_v1_path).num_rows == len(manifest)
assert pq.read_metadata(config.detector_v2_path).num_rows == len(manifest)
counts

In [ ]:
counts.plot(kind='bar', stacked=True, figsize=(8, 4), color=['#4472C4', '#70AD47', '#ED7D31'])
plt.title('Samples by source and split')
plt.xlabel('Source')
plt.ylabel('Messages')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()